# Flora & Fauna Image Classification — custom CNN from scratch

Multi-class image classifier for 10 taxonomic groups (animals, fungi and plants), built in PyTorch.

**Constraint:** no transfer learning and no transformer models. Every weight is randomly initialised and trained only on the provided data.

**Tip:** leave `DRY_RUN = True` in the Config cell for a ~2 minute end-to-end sanity check (tiny data slice, 1 epoch). Set it to `False` for the real ~35 minute training run on a GPU.

**Label mapping (fixed):**

| Class     | Label |
|-----------|-------|
| Amphibia  | 0 |
| Animalia  | 1 |
| Arachnida | 2 |
| Aves      | 3 |
| Fungi     | 4 |
| Insecta   | 5 |
| Mammalia  | 6 |
| Mollusca  | 7 |
| Plantae   | 8 |
| Reptilia  | 9 |

This is alphabetical order (the same order `ImageFolder` would assign), but the mapping is built **explicitly** so it stays correct regardless of folder ordering on disk.


In [1]:
# ===== Imports =====
import os, glob, random, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision
from torchvision import transforms
from torchvision.datasets import ImageFolder

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from PIL import Image

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


Torch: 2.10.0+cu128 | CUDA available: True


## 1. Config — fill in your details here

In [2]:
# ===== Config =====
DRY_RUN = False

SEED = 42
IMG_SIZE = 160          # bumped from 128 -- more detail helps distinguish visually similar
                        # classes (e.g. Arachnida vs Insecta). Costs ~1.5x epoch time.
BATCH_SIZE = 64
NUM_EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
VAL_SPLIT = 0.15
NUM_CLASSES = 10
PATIENCE = 12            # early stopping patience (epochs with no val-F1 improvement)
                          # kept generous because OneCycleLR is designed to run its full
                          # schedule -- stopping too early can cut off the LR annealing
                          # phase where a lot of the final gains happen.

if DRY_RUN:
    NUM_EPOCHS = 1
    print("DRY_RUN is ON: training will use a tiny data subset and 1 epoch, "
          "purely to check the pipeline runs end-to-end without errors. "
          "The resulting submission.csv will NOT be a real submission-quality file. "
          "Set DRY_RUN = False and re-run all cells for the actual training run.")

LABEL_MAP = {
    "Amphibia": 0, "Animalia": 1, "Arachnida": 2, "Aves": 3, "Fungi": 4,
    "Insecta": 5, "Mammalia": 6, "Mollusca": 7, "Plantae": 8, "Reptilia": 9,
}
IDX_TO_LABEL_NAME = {v: k for k, v in LABEL_MAP.items()}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 2. Locate the dataset

On Kaggle, the competition data is mounted read-only under `/kaggle/input/<competition-slug>/`.
The cell below auto-detects the folder that contains `train/` and `test/` so you don't have
to hardcode the competition slug. If auto-detection fails, set `DATA_DIR` manually.

In [3]:
# ===== Auto-detect data directory =====
# Searches a few levels deep under /kaggle/input, since Kaggle sometimes mounts
# competition data one level deeper (e.g. /kaggle/input/competitions/<slug>/)
# rather than directly under /kaggle/input/<slug>/.
DATA_DIR = None
search_roots = glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*")
for root in search_roots:
    if os.path.isdir(root) and os.path.isdir(os.path.join(root, "train")) \
            and os.path.isdir(os.path.join(root, "test")):
        DATA_DIR = root
        break

if DATA_DIR is None:
    # Fallback: set this manually to the competition data folder, e.g.
    # DATA_DIR = "/kaggle/input/competitions/<competition-slug>"
    raise FileNotFoundError(
        "Could not auto-detect DATA_DIR. Set it manually to the folder that directly "
        "contains 'train' and 'test' subfolders (check the 'Add Data' panel, or run "
        "`!find /kaggle/input -maxdepth 4 -iname train` in a cell to locate it)."
    )

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
print("DATA_DIR :", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

train_class_dirs = sorted([d for d in os.listdir(TRAIN_DIR)
                            if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print("\nTrain class folders found:", train_class_dirs)

test_entries = sorted(os.listdir(TEST_DIR))
test_subdirs = [e for e in test_entries if os.path.isdir(os.path.join(TEST_DIR, e))]
print("Entries directly inside TEST_DIR (first 5):", test_entries[:5])
if test_subdirs:
    print("\n*** WARNING: TEST_DIR contains SUBFOLDERS:", test_subdirs[:10], "***")
    print("This usually means TEST_DIR is pointing at the wrong folder (e.g. train "
          "instead of test). Fix DATA_DIR / TEST_DIR manually before continuing.")


DATA_DIR : /kaggle/input/competitions/competition-data
TRAIN_DIR: /kaggle/input/competitions/competition-data/train
TEST_DIR : /kaggle/input/competitions/competition-data/test

Train class folders found: ['Amphibia', 'Animalia', 'Arachnida', 'Aves', 'Fungi', 'Insecta', 'Mammalia', 'Mollusca', 'Plantae', 'Reptilia']
Entries directly inside TEST_DIR (first 5): ['Image_0001.jpg', 'Image_0002.jpg', 'Image_0003.jpg', 'Image_0004.jpg', 'Image_0005.jpg']


In [4]:
# ===== Sanity check: class counts =====
for cls in train_class_dirs:
    cls_path = os.path.join(TRAIN_DIR, cls)
    n = len(os.listdir(cls_path))
    print(f"{cls:12s} (label {LABEL_MAP.get(cls, '?')}): {n} images")

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Only keep actual FILES with an image extension -- this prevents a directory
# accidentally ending up in the test file list (which crashes Image.open with
# IsADirectoryError).
test_files = sorted([
    f for f in glob.glob(os.path.join(TEST_DIR, "*"))
    if os.path.isfile(f) and os.path.splitext(f)[1].lower() in IMG_EXTS
])
print(f"\nTest images found: {len(test_files)}")

assert len(test_files) > 0, (
    "No test image files found. Check that TEST_DIR is correct and that it "
    "contains image files directly (not nested in subfolders)."
)
assert all(os.path.isfile(f) for f in test_files), "Some entries in test_files are not files!"
print("Sample test file:", test_files[0])


Amphibia     (label 0): 1000 images
Animalia     (label 1): 1000 images
Arachnida    (label 2): 1000 images
Aves         (label 3): 1000 images
Fungi        (label 4): 999 images
Insecta      (label 5): 1000 images
Mammalia     (label 6): 1000 images
Mollusca     (label 7): 1000 images
Plantae      (label 8): 1000 images
Reptilia     (label 9): 1000 images

Test images found: 2000
Sample test file: /kaggle/input/competitions/competition-data/test/Image_0001.jpg


In [5]:
# In DRY_RUN mode, only run inference on a small slice of test images too,
# just to check the prediction loop and submission-writing code work correctly.
if DRY_RUN:
    test_files_for_run = test_files[:60]
    print("DRY_RUN: only predicting on", len(test_files_for_run), "test images "
          "(submission.csv will be INCOMPLETE -- do not submit it).")
else:
    test_files_for_run = test_files


## 3. Transforms & Datasets

In [6]:
# ===== Transforms =====
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # generic RGB normalization, no pretrained weights used
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


In [7]:
# ===== Custom Dataset that respects the fixed LABEL_MAP (not folder alphabetical order) =====
class FloraFaunaTrainDataset(Dataset):
    def __init__(self, root, label_map, transform=None):
        self.samples = []
        self.transform = transform
        for cls_name, label in label_map.items():
            cls_dir = os.path.join(root, cls_name)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                fpath = os.path.join(cls_dir, fname)
                if os.path.isfile(fpath):
                    self.samples.append((fpath, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


class FloraFaunaTestDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        image_id = os.path.splitext(os.path.basename(path))[0]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, image_id


full_train_dataset = FloraFaunaTrainDataset(TRAIN_DIR, LABEL_MAP, transform=None)
print("Total training images:", len(full_train_dataset))

labels_all = [lbl for _, lbl in full_train_dataset.samples]
train_idx, val_idx = train_test_split(
    np.arange(len(full_train_dataset)),
    test_size=VAL_SPLIT,
    random_state=SEED,
    stratify=labels_all,
)

if DRY_RUN:
    # Keep it tiny but still hit every class at least once where possible.
    train_idx = train_idx[:200]
    val_idx = val_idx[:60]
    print("DRY_RUN: using only", len(train_idx), "train and", len(val_idx), "val images.")

print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")


Total training images: 9999
Train: 8499 | Val: 1500


In [8]:
# ===== Wrap subsets with the right transform each =====
class TransformSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base_dataset = base_dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        path, label = self.base_dataset.samples[self.indices[i]]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset = TransformSubset(full_train_dataset, train_idx, train_transform)
val_dataset = TransformSubset(full_train_dataset, val_idx, eval_transform)
test_dataset = FloraFaunaTestDataset(test_files_for_run, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


Batches -> train: 132 val: 24 test: 32


## 4. Model — Custom CNN (built from scratch)

A VGG-style stack of conv blocks (Conv → BatchNorm → ReLU, repeated, then MaxPool),
increasing channel width, followed by global average pooling and a small classifier
head with dropout. No pretrained weights are loaded anywhere — every parameter is
randomly initialized and trained only on this competition's data.

In [9]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation: learns per-channel importance weights.
    Small, cheap, and gives a consistent bump for from-scratch CNNs."""
    def __init__(self, channels, reduction=8):
        super().__init__()
        reduced = max(channels // reduction, 4)
        self.fc1 = nn.Linear(channels, reduced)
        self.fc2 = nn.Linear(reduced, channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        s = x.mean(dim=(2, 3))                 # global average pool -> (b, c)
        s = F.relu(self.fc1(s))
        s = torch.sigmoid(self.fc2(s))          # (b, c) channel weights in [0,1]
        return x * s.view(b, c, 1, 1)


class ConvBlock(nn.Module):
    """Conv block with a residual (skip) connection + SE attention: helps gradient
    flow and lets the network learn which channels matter, without any pretrained
    weights."""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.se = SEBlock(out_ch)
        # 1x1 projection so the skip connection matches out_ch when in_ch != out_ch
        self.skip = (nn.Conv2d(in_ch, out_ch, kernel_size=1)
                     if in_ch != out_ch else nn.Identity())
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = F.relu(out + identity)
        out = self.pool(out)
        return out


class FloraFaunaCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()
        self.block1 = ConvBlock(3, 32)      # -> IMG/2
        self.block2 = ConvBlock(32, 64)     # -> IMG/4
        self.block3 = ConvBlock(64, 128)    # -> IMG/8
        self.block4 = ConvBlock(128, 256)   # -> IMG/16
        self.block5 = ConvBlock(256, 256, pool=False)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.block5(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


model = FloraFaunaCNN(num_classes=NUM_CLASSES).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")


class EMA:
    """Exponential Moving Average of model weights. Keeping a smoothed copy of the
    weights and using it at inference time usually gives a small, consistent F1
    bump over the raw final-epoch weights, almost for free."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model.state_dict())

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v, alpha=1 - self.decay)
            else:
                self.shadow[k] = v.clone()

    def state_dict(self):
        return self.shadow


ema = EMA(model, decay=0.999)


Trainable parameters: 2,506,278


## 5. Training setup + training loop (kept in one cell — see note below)

In [10]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# OneCycleLR: ramps LR up then anneals down smoothly over the WHOLE training run --
# tends to converge faster/higher than ReduceLROnPlateau for a from-scratch CNN with
# a fixed epoch budget. It is stepped once per BATCH (not per epoch).
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR * 3, epochs=NUM_EPOCHS, steps_per_epoch=len(train_loader),
    pct_start=0.25
)

# NOTE: this cell MUST be run (and stay defined) before later cells that use it.
# If your kernel ever restarts, use "Run All" rather than running a later cell
# in isolation, or criterion/optimizer/run_epoch will be undefined.

def run_epoch(loader, model, criterion, optimizer=None, scheduler=None,
              ema=None, scaler=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    torch.set_grad_enabled(is_train)
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if is_train:
            optimizer.zero_grad()
            with torch.autocast(device_type="cuda", enabled=(scaler is not None)):
                outputs = model(imgs)
                loss = criterion(outputs, labels)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
            if scheduler is not None:
                scheduler.step()   # OneCycleLR steps per batch, not per epoch
            if ema is not None:
                ema.update(model)
        else:
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average="micro")
    return avg_loss, f1


# ===== 6. Training loop (with early stopping on validation F1) =====
# Kept in the SAME cell as the setup above on purpose: if the kernel ever restarts,
# re-running just this one cell is enough to redefine criterion/optimizer/run_epoch
# AND run training, instead of hitting a NameError from a skipped earlier cell.

# Mixed precision: same accuracy, faster training -- buys back the time cost of
# IMG_SIZE=160 and lets you fit more epochs in the same GPU budget.
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# A second model instance used only to evaluate the EMA (smoothed) weights each
# epoch, so we can compare "raw" vs "EMA" validation F1 and keep whichever is better.
ema_eval_model = FloraFaunaCNN(num_classes=NUM_CLASSES).to(device)

history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [], "ema_val_f1": []}
best_val_f1 = -1.0
best_state = None
epochs_no_improve = 0
start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_f1 = run_epoch(train_loader, model, criterion, optimizer,
                                      scheduler, ema=ema, scaler=scaler)
    val_loss, val_f1 = run_epoch(val_loader, model, criterion, optimizer=None)

    ema_eval_model.load_state_dict(ema.state_dict())
    _, ema_val_f1 = run_epoch(val_loader, ema_eval_model, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)
    history["ema_val_f1"].append(ema_val_f1)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss {train_loss:.4f} train_f1 {train_f1:.4f} | "
          f"val_f1 {val_f1:.4f} | ema_val_f1 {ema_val_f1:.4f} | lr {current_lr:.2e}")

    # Track whichever is currently better: raw weights or EMA weights.
    best_this_epoch = max(val_f1, ema_val_f1)
    if best_this_epoch > best_val_f1:
        best_val_f1 = best_this_epoch
        best_state = copy.deepcopy(ema.state_dict() if ema_val_f1 >= val_f1
                                    else model.state_dict())
        epochs_no_improve = 0
        torch.save(best_state, "best_model.pth")
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch} (no val F1 improvement for {PATIENCE} epochs).")
        break

elapsed = time.time() - start_time
print(f"\nTraining finished in {elapsed/60:.1f} min. Best val F1 (micro): {best_val_f1:.4f}")

model.load_state_dict(best_state)


/tmp/ipykernel_23/2853562541.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


Epoch 01/30 | train_loss 2.1581 train_f1 0.2126 | val_f1 0.2673 | ema_val_f1 0.1000 | lr 2.45e-04
Epoch 02/30 | train_loss 2.1019 train_f1 0.2475 | val_f1 0.2620 | ema_val_f1 0.0993 | lr 5.97e-04
Epoch 03/30 | train_loss 2.0794 train_f1 0.2624 | val_f1 0.2747 | ema_val_f1 0.1093 | lr 1.12e-03
Epoch 04/30 | train_loss 2.0701 train_f1 0.2654 | val_f1 0.2640 | ema_val_f1 0.1160 | lr 1.71e-03
Epoch 05/30 | train_loss 2.0466 train_f1 0.2660 | val_f1 0.2747 | ema_val_f1 0.1100 | lr 2.28e-03
Epoch 06/30 | train_loss 2.0298 train_f1 0.2794 | val_f1 0.2593 | ema_val_f1 0.1013 | lr 2.73e-03
Epoch 07/30 | train_loss 2.0150 train_f1 0.2875 | val_f1 0.2507 | ema_val_f1 0.1053 | lr 2.97e-03
Epoch 08/30 | train_loss 1.9843 train_f1 0.2990 | val_f1 0.2867 | ema_val_f1 0.1240 | lr 3.00e-03
Epoch 09/30 | train_loss 1.9524 train_f1 0.3176 | val_f1 0.2580 | ema_val_f1 0.1527 | lr 2.97e-03
Epoch 10/30 | train_loss 1.9223 train_f1 0.3314 | val_f1 0.3267 | ema_val_f1 0.1707 | lr 2.91e-03
Epoch 11/30 | train_

<All keys matched successfully>

## 7. Training curves

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_f1"], label="train")
axes[1].plot(history["val_f1"], label="val (raw)")
axes[1].plot(history["ema_val_f1"], label="val (EMA)")
axes[1].set_title("F1 (micro)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


<Figure size 1200x400 with 2 Axes>

![Training curves](../assets/training_curves.png)


## 7b. Confusion matrix on validation set

See exactly which classes the model confuses, so you know where to focus next
(e.g. more targeted augmentation, or accepting that two classes are just visually
similar and there's a ceiling on separating them).

In [12]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()
val_preds, val_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        val_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        val_labels.extend(labels.numpy())

class_names = [IDX_TO_LABEL_NAME[i] for i in range(NUM_CLASSES)]
print(classification_report(val_labels, val_preds, target_names=class_names, digits=3))

cm = confusion_matrix(val_labels, val_preds)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
plt.colorbar(im)
plt.tight_layout()
plt.show()


              precision    recall  f1-score   support

    Amphibia      0.374     0.433     0.401       150
    Animalia      0.500     0.520     0.510       150
   Arachnida      0.630     0.567     0.596       150
        Aves      0.506     0.593     0.546       150
       Fungi      0.566     0.600     0.583       150
     Insecta      0.496     0.453     0.474       150
    Mammalia      0.479     0.447     0.462       150
    Mollusca      0.523     0.373     0.436       150
     Plantae      0.568     0.640     0.602       150
    Reptilia      0.354     0.347     0.350       150

    accuracy                          0.497      1500
   macro avg      0.499     0.497     0.496      1500
weighted avg      0.499     0.497     0.496      1500



<Figure size 800x700 with 2 Axes>

![Confusion matrix](../assets/confusion_matrix.png)


## 8. Inference on the test set & building the submission file

In [13]:
# Test-Time Augmentation (TTA): average the softmax probabilities of each image and
# its horizontal flip. Free accuracy boost, ~2x inference time (still well under a
# minute for 2000 test images).
model.eval()
image_ids, preds = [], []

with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)
        logits_orig = model(imgs)
        logits_flip = model(torch.flip(imgs, dims=[3]))  # horizontal flip
        probs = F.softmax(logits_orig, dim=1) + F.softmax(logits_flip, dim=1)
        batch_preds = probs.argmax(dim=1).cpu().numpy()
        preds.extend(batch_preds.tolist())
        image_ids.extend(ids)

submission = pd.DataFrame({"Image_ID": image_ids, "Label": preds})

# Keep the same row order as sample_submission.csv if available, for safety
sample_path = os.path.join(DATA_DIR, "sample_submission.csv")
if os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    id_col = sample.columns[0]
    submission = sample[[id_col]].merge(
        submission, left_on=id_col, right_on="Image_ID", how="left"
    )[["Image_ID", "Label"]]

if DRY_RUN:
    print("DRY_RUN: skipping full row-count assertions -- this submission.csv is a "
          "format check only, not something to submit to Kaggle.")
    submission.to_csv("submission_dry_run.csv", index=False)
    print("Saved submission_dry_run.csv with", len(submission), "rows. Columns/dtypes look like:")
    print(submission.dtypes)
else:
    assert submission["Label"].isna().sum() == 0, "Some test images were not predicted!"
    assert len(submission) == len(test_files), "Row count mismatch with test set!"
    submission.to_csv("submission.csv", index=False)
    print("Saved submission.csv with", len(submission), "rows")

submission.head()


Saved submission.csv with 2000 rows


,Image_ID,Label
0,Image_0001,0
1,Image_0002,4
2,Image_0003,5
3,Image_0004,1
4,Image_0005,4


## 9. Notes

- The model is a randomly-initialised custom CNN (residual conv blocks + squeeze-and-excitation). No pretrained backbone and no transformer layers are used.
- `IMAGENET_MEAN` / `IMAGENET_STD` are just generic RGB normalisation constants; no pretrained weights are loaded.
- The run of record reached a best validation micro-F1 of about 0.497 (see `README.md` for details).
